In [2]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import Questions
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind
from itertools import combinations

In [3]:
# definitions
models = [
    "gpt_4o",
    "llama31_70b_Inst",
    "qwen25_72b_Inst",
    "llama31_8B_Inst",
    "Qwen25_7B_Inst",
    "llama32_1B_Inst",
    "Qwen25_1dot5B_Inst"
]

demography_profiles = [
    "employment_sector",
    "race",
    "education",
    "sex",
    "age",
    "language",
    "nationality_group"
]
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
EXCLUDED_SCORES = ["2_13", "23_", "24_", "25_", "27_"]

questions = Questions().get_questions()

task_group_dict = {'qa_extraction': [], 'semantics': [], 'mt_and_summarization': [], 'syntaxes': [], 'knowledge_and_reasoning': []}
for i, k in enumerate(questions.keys()):
    task_group_dict[questions[k]["question_type"]].append(i)

stacked_model_df = []
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]
    stacked_model_df.append(score_df[task_cols])

def get_min_max_from_stacked_df(stacked_model_df):
    # Stack all DataFrames into one
    combined_df = pd.concat(stacked_model_df, ignore_index=True)

    # Compute min and max for each column
    return {
        col: {
            'min': combined_df[col].min(),
            'max': combined_df[col].max()
        }
        for col in combined_df.columns
    }

universal_scoredf_minmax = get_min_max_from_stacked_df(stacked_model_df)

scorelists_by_model = {}
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]

    fullscores_per_dp = {}
    for dp in demography_profiles:
        fullscores_per_dp[dp] = {}
        for tc in task_cols:
            fullscores_per_dp[dp][tc] = {}
            if not any(tc.startswith(prefix) for prefix in EXCLUDED_SCORES):
                demo_grouped = score_df.groupby([dp])
                for demo_name, group in demo_grouped:
                    # implement here for exceptions
                    if dp == "employment_sector":
                        group = group[group["language"] == "L1"]
                    elif dp == "education":
                        group = group[group["language"] == "L1"]
                    elif dp == "sex":
                        group = group[group["language"] == "L1"]
                        
                    elif(demo_name == "Black"):
                        group = group[group["language"] == "L1"]
                    elif(demo_name == "White"):
                        group = group[group["language"] == "L1"]
                    # implement here for exceptions
                    fullscores_per_dp[dp][tc][demo_name[0]] = group[tc].dropna().to_list()
    scorelists_by_model[model] = fullscores_per_dp

In [11]:
print(scorelists_by_model['gpt_4o'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['0_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].values())

dict_keys(['employment_sector', 'race', 'education', 'sex', 'age', 'language', 'nationality_group'])
dict_keys(['0_0', '1_0', '2_11', '2_3', '2_4', '2_10', '2_5', '2_2', '2_1', '2_6', '2_8', '2_14', '2_13', '2_9', '2_7', '2_0', '2_12', '3_0', '4_0', '6_0', '8_0', '9_0', '11_0', '12_0', '13_0', '14_0', '15_0', '16_0', '17_0', '18_0', '19_0', '20_0', '21_0', '22_0', '26_0', '26_1', '5_0', '7_0', '10_0', '28_0'])
dict_keys(['Engineering-STEM', 'Health-STEM', 'Humanities'])
dict_keys(['Engineering-STEM', 'Health-STEM', 'Humanities'])
dict_values([[0.9137484905985856, 0.9236874236874236, 0.8915558126084442, 0.9177601887102792, 0.9077505575075002, 0.9238525785921544, 0.9499276955924356, 0.935897435897436, 0.933806981157054, 0.9519377112738108, 0.9338833702651476, 0.911682055399438, 0.923760914227016, 0.9318429661941112, 0.9297637251714768, 0.937844611528822], [0.9076676033721396, 0.901792192278862, 0.8552496139989707, 0.9216990109982774, 0.9178814207715942, 0.939952923091704, 0.9339236156997

In [ ]:
import math
from scipy.stats import f_oneway

def get_minmax_vals(data_dict):
    all_values = [x for group in data_dict.values() for x in group]
    return min(all_values), max(all_values)

def compute_diversity_score(data_dict):
    if len(data_dict) < 2:
        return 0  # No diversity if less than 2 elements
    
    min_val, max_val = get_minmax_vals(data_dict)

    list_of_values = []
    for lv in data_dict.values():
        if max_val == min_val:
            # All values are the same → treat as constant
            avg = 0.5  # or any neutral default
        else:
            normalized = [(x - min_val) / (max_val - min_val) for x in lv]
            avg = sum(normalized) / len(normalized)
        list_of_values.append(avg)
    
    total = []
    for i in range(len(list_of_values)):
        for j in range(i + 1, len(list_of_values)):
            total.append(abs(list_of_values[i] - list_of_values[j]))

    return sum(total) / len(total)

def compute_anova_score(data_dict):
    # normalizer
    min_val, max_val = get_minmax_vals(data_dict)
    
    if max_val == min_val:
        # All values are the same across all groups
        return float("nan"), float("nan")

    groups = []
    for scores in data_dict.values():
        norm = [(x - min_val) / (max_val - min_val) for x in scores]
        groups.append(norm)
    
    # Perform one-way ANOVA
    f_statistic, p_value = f_oneway(*groups)
    return f_statistic, p_value


In [23]:
for tg_name, tg_idxs in task_group_dict.items():
    divscore_obj_list = []
    for modl in models:
        table_row = {"model" : modl}
        for k in scorelists_by_model[modl].keys():
            tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
            tasks = [qdi for qdi in tasks if int(qdi.split("_")[0]) in tg_idxs]

            divsc_total = 0
            for qdi in tasks:
                divsc_total += compute_diversity_score(scorelists_by_model[modl][k][qdi])            
            table_row[k] = divsc_total/len(tasks)

        divscore_obj_list.append(table_row)
    pd.DataFrame(divscore_obj_list).to_csv(f"div_score_tables/divscores_{tg_name}.csv")

In [55]:
pd.DataFrame(divscore_obj_list)

,model,employment_sector,race,education,sex,age,language,nationality_group
0,gpt_4o,0.026771,0.009695,0.016868,0.023964,0.013111,0.007141,0.021549
1,llama31_70b_Inst,0.020897,0.036937,0.015145,0.011881,0.025040,0.061320,0.037218
2,qwen25_72b_Inst,0.018022,0.013205,0.022116,0.010913,0.013854,0.014756,0.021158
3,llama31_8B_Inst,0.012507,0.021739,0.014419,0.009971,0.014257,0.014984,0.019491
4,Qwen25_7B_Inst,0.005646,0.016338,0.015756,0.008442,0.009942,0.015755,0.013725
5,llama32_1B_Inst,0.043509,0.075531,0.043741,0.030181,0.021466,0.093744,0.068608
6,Qwen25_1dot5B_Inst,0.031774,0.043445,0.031981,0.016988,0.026792,0.026551,0.038674


In [ ]:
# ANOVA P VALUE TESTS
REJECT_THRESHOLD = 0.05

ftest_obj_list = []
for modl in models:
    table_row = {"model" : modl}
    for k in scorelists_by_model[modl].keys():
        ftest_list=[]
        tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
        task_count = len(tasks)
        for qdi in tasks:
            f, p = compute_anova_score(scorelists_by_model[modl][k][qdi])
            ftest_list.append(f)
            # adj_p = p * task_count * len(models)
            adj_p = p * task_count
            if (adj_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task-{qdi}", k, adj_p)
        ftest_filtered = [x for x in ftest_list if not math.isnan(x)]
        table_row[k] = sum(ftest_filtered) / len(ftest_filtered) if ftest_filtered else None
    ftest_obj_list.append(table_row)

ftest_df = pd.DataFrame(ftest_obj_list)
numeric_df = ftest_df.drop(columns=['model'])
row_means = numeric_df.mean(axis=1)
ftest_df['MODEL WISE MEAN'] = row_means

column_means = numeric_df.mean()
mean_row = column_means.copy()
mean_row['model'] = 'DEMOGRAPHY MEAN'
mean_row['MODEL WISE MEAN'] = float("nan")

df = pd.concat([ftest_df, pd.DataFrame([mean_row])], ignore_index=True)
df

llama31_70b_Inst REJECTED! task-28_0 age 0.014737161360797392
llama31_70b_Inst REJECTED! task-20_0 language 0.033048407179910026
Qwen25_7B_Inst REJECTED! task-26_1 age 0.009772178920714998
llama32_1B_Inst REJECTED! task-21_0 race 0.021606535354551728
llama32_1B_Inst REJECTED! task-21_0 language 0.005837598627334997
llama32_1B_Inst REJECTED! task-26_1 language 0.019001959319213307
llama32_1B_Inst REJECTED! task-21_0 nationality_group 0.04440016279410678


,model,employment_sector,race,education,sex,age,language,nationality_group,MODEL WISE MEAN
0,gpt_4o,0.938733,1.059700,0.997535,1.255356,1.232642,0.824975,1.154868,1.066258
1,llama31_70b_Inst,1.282116,1.790050,1.187866,0.573854,1.369606,1.539203,1.662821,1.343645
2,qwen25_72b_Inst,1.001818,1.043760,1.086093,1.218508,1.089574,1.291942,1.171789,1.129069
3,llama31_8B_Inst,0.867621,1.341729,1.174878,1.059561,1.416391,1.531551,1.844491,1.319460
4,Qwen25_7B_Inst,1.180106,1.137426,1.382765,0.965277,1.434255,1.742754,1.026724,1.267044
5,llama32_1B_Inst,0.910052,1.405647,0.961751,0.711733,1.118108,2.003189,1.550193,1.237239
6,Qwen25_1dot5B_Inst,1.167202,1.304088,0.796284,1.061088,0.968358,1.497051,1.395094,1.169881
7,DEMOGRAPHY MEAN,1.049664,1.297486,1.083881,0.977911,1.232705,1.490095,1.400854,NaN


In [6]:
df

,model,employment_sector,race,education,sex,age,language,nationality_group,MODEL WISE MEAN
0,gpt_4o,0.938733,1.059700,0.997535,1.255356,1.232642,0.824975,1.154868,1.066258
1,llama31_70b_Inst,1.282116,1.790050,1.187866,0.573854,1.369606,1.539203,1.662821,1.343645
2,qwen25_72b_Inst,1.001818,1.043760,1.086093,1.218508,1.089574,1.291942,1.171789,1.129069
3,llama31_8B_Inst,0.867621,1.341729,1.174878,1.059561,1.416391,1.531551,1.844491,1.319460
4,Qwen25_7B_Inst,1.180106,1.137426,1.382765,0.965277,1.434255,1.742754,1.026724,1.267044
5,llama32_1B_Inst,0.910052,1.405647,0.961751,0.711733,1.118108,2.003189,1.550193,1.237239
6,Qwen25_1dot5B_Inst,1.167202,1.304088,0.796284,1.061088,0.968358,1.497051,1.395094,1.169881
7,DEMOGRAPHY MEAN,1.049664,1.297486,1.083881,0.977911,1.232705,1.490095,1.400854,NaN


In [ ]:

mean_row

employment_sector           1.049664
race                        1.297486
education                   1.083881
sex                         0.977911
age                         1.232705
language                    1.490095
nationality_group           1.400854
model                DEMOGRAPHY MEAN
MODEL WISE MEAN                  NaN
dtype: object

In [5]:
def df_to_latex_percent(df):
    def format_val(val):
        if isinstance(val, float):
            return f"{round(val * 100, 1)}\\%"
        return str(val).replace("_", "\\_")

    formatted_df = df.copy()
    formatted_df.columns = [col.replace("_", "\\_") for col in df.columns]
    for col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].apply(format_val)

    return formatted_df.to_latex(index=False, escape=False)

In [6]:
print(df_to_latex_percent(demo_pvalue_perquestion))

\begin{tabular}{llllllll}
\toprule
Model Name & employment\_sector & race & education & sex & age & language & nationality\_group \\
\midrule
gpt\_4o & 7.7\% & 17.9\% & 7.7\% & 5.1\% & 15.4\% & 5.1\% & 20.5\% \\
llama31\_70B\_Inst & 15.4\% & 23.1\% & 7.7\% & 0.0\% & 17.9\% & 10.3\% & 38.5\% \\
Qwen25\_72B\_Inst & 5.1\% & 10.3\% & 5.1\% & 7.7\% & 5.1\% & 7.7\% & 23.1\% \\
llama31\_8B\_Inst & 20.5\% & 17.9\% & 5.1\% & 5.1\% & 20.5\% & 12.8\% & 30.8\% \\
Qwen25\_7B\_Inst & 12.8\% & 23.1\% & 25.6\% & 7.7\% & 10.3\% & 15.4\% & 28.2\% \\
llama32\_1B\_Inst & 7.7\% & 23.1\% & 2.6\% & 5.1\% & 7.7\% & 2.6\% & 30.8\% \\
Qwen25\_1dot5B\_Inst & 20.5\% & 17.9\% & 5.1\% & 0.0\% & 15.4\% & 23.1\% & 30.8\% \\
\bottomrule
\end{tabular}



In [7]:
print(df_to_latex_percent(sig_diversity_scores))

\begin{tabular}{llllllll}
\toprule
Model Name & employment\_sector & race & education & sex & age & language & nationality\_group \\
\midrule
gpt\_4o & 13.0\% & 11.3\% & 5.0\% & 3.7\% & 6.8\% & 10.3\% & 6.3\% \\
llama31\_70B\_Inst & 5.1\% & 8.7\% & 11.7\% & nan\% & 8.7\% & 8.4\% & 13.4\% \\
Qwen25\_72B\_Inst & 5.5\% & 14.4\% & 4.4\% & 10.7\% & 21.0\% & 16.3\% & 8.1\% \\
llama31\_8B\_Inst & 9.2\% & 8.9\% & 30.6\% & 9.2\% & 7.1\% & 12.4\% & 10.1\% \\
Qwen25\_7B\_Inst & 8.6\% & 6.4\% & 13.6\% & 6.3\% & 6.3\% & 6.3\% & 8.7\% \\
llama32\_1B\_Inst & 15.8\% & 9.7\% & 14.1\% & 7.7\% & 10.4\% & 13.3\% & 10.9\% \\
Qwen25\_1dot5B\_Inst & 8.3\% & 6.9\% & 19.1\% & nan\% & 5.7\% & 7.0\% & 7.3\% \\
\bottomrule
\end{tabular}

